# Práctica 6 Web Scraping

In [2]:
print("Hello, World!")

Hello, World!


En este cuaderno aprenderemos a construir un flujo de Web Scraping para transformar páginas web en conjuntos de datos estructurados y listos para su análisis.

Antes de integrar herramientas de automatización como GitHub Actions o plataformas de visualización como Power BI, el primer paso fundamental ocurre en el entorno de desarrollo: inspeccionar, comprender y extraer la información directamente desde el código fuente del sitio objetivo.

A lo largo de este ejercicio práctico aprenderemos a:

* **Inspeccionar páginas web dinámicas:** Utilizar las herramientas de desarrollador (F12 / DevTools) para analizar la estructura HTML de cualquier sitio y seleccionar los contenedores clave sin depender de plantillas rígidas.

* **Procesar y limpiar datos con Python:** Automatizar la recolección masiva mediante ciclos, limpiar cadenas de texto no deseadas (.text, .strip(), .replace()) y organizar la información en tablas estructuradas utilizando la librería Pandas.

* **Exportar resultados para BI:** Generar archivos CSV estandarizados que servirán como la fuente primaria de datos para la posterior automatización y conexión a tableros de control.

En esta versión trabajaremos con la página de Wikipedia **"List of best-selling video games"**, que contiene una tabla real (no un sandbox) con los videojuegos más vendidos de la historia: título, ventas, serie, plataformas, año de lanzamiento, desarrollador y editor.


In [1]:
# Requests sirve para enviar solicitudes HTTP a servidores web, permite interactuar con páginas web.
import requests

# BeautifulSoup sirve para analizar y extraer datos de documentos HTML y XML.
from bs4 import BeautifulSoup

import pandas as pd

In [2]:
# 1. Hacer la petición a la página objetivo
url = "https://en.wikipedia.org/wiki/List_of_best-selling_video_games"

# Wikipedia requiere un User-Agent identificable; sin este encabezado puede rechazar la petición.
headers = {"User-Agent": "Mozilla/5.0 (compatible; CursoScraping/1.0)"}

#Envía una petición GET al servidor. El servidor entrega todo el HTML de la página.
response = requests.get(url, headers=headers)

# Garantiza que caracteres especiales se interpreten sin errores.
response.encoding = 'utf-8'

In [3]:
# 2. Convertir el texto HTML a un objeto parseable con BeautifulSoup
# Toma el texto HTML de la respuesta y lo analiza con el motor "html.parser".
# Un objeto parseable es un dato en formato de texto plano que tiene la estructura correcta para ser analizado
texto = BeautifulSoup(response.text, "html.parser")
texto

<!DOCTYPE html>

<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" dir="ltr" lang="en">
<head>
<meta charset="utf-8"/>
<title>List of best-selling video games - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-p

In [4]:
# 3. Localizar la tabla de videojuegos más vendidos (etiqueta <table class="wikitable">)
# Busca en todo el documento HTML la primera tabla que cumpla la condición.
# find() (a diferencia de find_all) devuelve solo la primera coincidencia, que es la tabla que nos interesa.

tabla = texto.find("table", class_="wikitable")

# find_all("tr") localiza todas las filas de la tabla; con [1:] descartamos la fila de encabezado.
filas = tabla.find_all("tr")[1:]
filas

[<tr id="mweA">
 <td id="mweQ" rowspan="2" style="text-align: center;">1<sup about="#mwt117" class="mw-ref reference" data-mw='{"name":"ref","attrs":{"name":"Dispute","group":"lower-alpha"},"body":{"html":""},"parts":[{"template":{"target":{"wt":"Efn","href":"./Template:Efn"},"params":{"name":{"wt":"Dispute"}},"i":0}}]}' id="cite_ref-Dispute_11-1" rel="dc:references" typeof="mw:Transclusion mw:Extension/ref"><a data-mw-group="lower-alpha" href="#cite_note-Dispute-11" id="mweg"><span class="mw-reflink-text" id="mwew"><span class="cite-bracket" id="mwfA">[</span>a<span class="cite-bracket" id="mwfQ">]</span></span></a></sup></td>
 <th id="mwfg" scope="row"><i id="mwfw"><a href="https://en.wikipedia.org/wiki/Tetris" id="mwgA" rel="mw:WikiLink" title="Tetris">Tetris</a></i></th>
 <td id="mwgQ" style="text-align: center;">520</td>
 <td id="mwgg">None</td>
 <td id="mwgw">Multi-platform</td>
 <td id="mwhA">1988<sup about="#mwt120" class="mw-ref reference" data-mw="{&quot;name&quot;:&quot;ref&

In [5]:
datos = []

# 4. Iterar sobre cada fila para "pescar" los datos sueltos
for fila in filas:
    celdas = fila.find_all(["td", "th"])

    # Algunas filas no incluyen la columna "Rank" porque está fusionada (rowspan) con la fila anterior
    # (ocurre cuando dos juegos empatan en el mismo puesto). Por eso tomamos siempre las últimas 8 celdas,
    # que son las que sí están garantizadas en cada fila: Título, Ventas, Serie, Plataforma(s),
    # Año, Desarrollador(es), Editor(es) y Referencia.
    celdas = celdas[-8:]

    titulo = celdas[0].get_text(strip=True)
    ventas_millones = celdas[1].get_text(strip=True)
    serie = celdas[2].get_text(strip=True)
    plataformas = celdas[3].get_text(strip=True)
    anio_lanzamiento = celdas[4].get_text(strip=True)
    desarrollador = celdas[5].get_text(strip=True)
    editor = celdas[6].get_text(strip=True)

    datos.append({
        "titulo": titulo,
        "ventas_millones": ventas_millones,
        "serie": serie,
        "plataformas": plataformas,
        "anio_lanzamiento": anio_lanzamiento,
        "desarrollador": desarrollador,
        "editor": editor
    })

In [6]:
# 5. Convertir la lista de diccionarios a un DataFrame (Estructura Tidy Data)
df = pd.DataFrame(datos)
df

,titulo,ventas_millones,serie,plataformas,anio_lanzamiento,desarrollador,editor
0,Tetris,520,None,Multi-platform,1988[c],Various,Various
1,Minecraft,400,Minecraft,Multi-platform,2011[d],Mojang Studios,Mojang Studios
2,Grand Theft Auto V,230,Grand Theft Auto,Multi-platform,2013,Rockstar North,Rockstar Games
3,Red Dead Redemption 2,87,Red Dead,Multi-platform,2018,Rockstar Games,Rockstar Games
4,Wii Sports[b],82.9,Wii,Wii,2006,Nintendo EAD,Nintendo
5,Mario Kart 8/Deluxe,79.99,Mario Kart,Wii U/Switch,2014,Nintendo EAD/Nintendo EPD(Deluxe),Nintendo
6,PUBG: Battlegrounds,75,PUBG Universe,Multi-platform,2017,PUBG Studios,Krafton
7,Terraria,70,None,Multi-platform,2011,Re-Logic,Re-Logic/505 Games
8,The Elder Scrolls V: Skyrim,65,The Elder Scrolls,Multi-platform,2011,Bethesda Game Studios,Bethesda Softworks
9,The Witcher 3: Wild Hunt,65,The Witcher,Multi-platform,2015,CD Projekt Red,CD Projekt


In [7]:
%%writefile scraper.py
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://en.wikipedia.org/wiki/List_of_best-selling_video_games"
headers = {"User-Agent": "Mozilla/5.0 (compatible; CursoScraping/1.0)"}
response = requests.get(url, headers=headers)
response.encoding = 'utf-8'

soup = BeautifulSoup(response.text, "html.parser")
tabla = soup.find("table", class_="wikitable")
filas = tabla.find_all("tr")[1:]

datos = []
for fila in filas:
    celdas = fila.find_all(["td", "th"])[-8:]

    datos.append({
        "titulo": celdas[0].get_text(strip=True),
        "ventas_millones": celdas[1].get_text(strip=True),
        "serie": celdas[2].get_text(strip=True),
        "plataformas": celdas[3].get_text(strip=True),
        "anio_lanzamiento": celdas[4].get_text(strip=True),
        "desarrollador": celdas[5].get_text(strip=True),
        "editor": celdas[6].get_text(strip=True)
    })

df = pd.DataFrame(datos)
df.to_csv("videojuegos_mas_vendidos.csv", index=False)
print("Scraping exitoso y archivo videojuegos_mas_vendidos.csv creado.")

Writing scraper.py
